In [ ]:
## RUN THESE IF YOU ARE ON COLAB
# !git clone https://github.com/anilegin/nlp-sentenceSplitter.git
# !cd nlp-sentenceSplitter/
# import sys
# sys.path.append('/content/nlp-sentenceSplitter')

Cloning into 'nlp-sentenceSplitter'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 135 (delta 37), reused 109 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 10.20 MiB | 14.84 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import classification_report, f1_score

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
try:
    train_df = pd.read_csv("/content/nlp-sentenceSplitter/data/processed/UD_English-EWT/en_ewt-ud-train.csv")
    dev_df = pd.read_csv("/content/nlp-sentenceSplitter/data/processed/UD_English-EWT/en_ewt-ud-dev.csv")
except FileNotFoundError:
    train_df = pd.read_csv("./data/processed/UD_English-EWT/en_ewt-ud-train.csv")
    dev_df = pd.read_csv("./data/processed/UD_English-EWT/en_ewt-ud-dev.csv")
    raise

print(train_df.shape, dev_df.shape)
print(train_df["label"].value_counts())
print(dev_df["label"].value_counts())

(30337, 9) (5035, 9)
label
0    18333
1    12004
Name: count, dtype: int64
label
0    3120
1    1915
Name: count, dtype: int64


In [5]:
from utils.featureExtractor import FeatureExtractor

feat_extractor = FeatureExtractor()

X_train_feat = feat_extractor.fit_transform(train_df)
X_dev_feat = feat_extractor.transform(dev_df)

print(X_train_feat.shape, X_dev_feat.shape)
X_train_feat.head()

(30337, 38) (5035, 38)


,is_period,is_question,is_exclamation,is_newline_candidate,prev_is_digit,next_is_digit,prev_is_alpha,next_is_alpha,prev_is_space,next_is_space,...,leading_closing_brackets_after,has_quote_after,has_closing_bracket_after,prev_is_punct,next_is_punct,trailing_punct_before,leading_punct_after,newline_in_left_10,newline_in_right_10,double_newline_in_right_20
0,0,0,0,1,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,1,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,1,0,0,0,...,1,0,1,0,0,0,0,0,0,0
4,0,0,0,1,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
def build_char_vocab(texts, min_freq=1):
    freq = {}

    for text in texts.fillna("").astype(str):
        for ch in text:
            freq[ch] = freq.get(ch, 0) + 1

    chars = [ch for ch, c in freq.items() if c >= min_freq]
    chars = sorted(chars)

    stoi = {"<pad>": 0, "<unk>": 1}
    for ch in chars:
        stoi[ch] = len(stoi)

    itos = {idx: ch for ch, idx in stoi.items()}
    return stoi, itos

In [7]:
stoi, itos = build_char_vocab(train_df["centered_context"], min_freq=1)
print("vocab size:", len(stoi))

vocab size: 112


In [8]:
def encode_text(text, stoi, max_len):
    text = "" if pd.isna(text) else str(text)
    ids = [stoi.get(ch, stoi["<unk>"]) for ch in text[:max_len]]

    if len(ids) < max_len:
        ids += [stoi["<pad>"]] * (max_len - len(ids))

    return ids

MAX_LEN = int(train_df["centered_context"].fillna("").astype(str).str.len().quantile(0.99))
MAX_LEN = max(64, min(MAX_LEN, 160))
print("MAX_LEN:", MAX_LEN)

MAX_LEN: 101


In [9]:
class CharCNNDataset(Dataset):
    def __init__(self, df, stoi, max_len, numeric_features=None):
        self.texts = df["centered_context"].fillna("").astype(str).tolist()
        self.labels = df["label"].astype(np.float32).tolist()
        self.stoi = stoi
        self.max_len = max_len

        self.numeric_features = None
        if numeric_features is not None:
            self.numeric_features = numeric_features.astype(np.float32).values

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x_char = torch.tensor(
            encode_text(self.texts[idx], self.stoi, self.max_len),
            dtype=torch.long
        )
        y = torch.tensor(self.labels[idx], dtype=torch.float32)

        if self.numeric_features is not None:
            x_num = torch.tensor(self.numeric_features[idx], dtype=torch.float32)
            return x_char, x_num, y

        return x_char, y

In [10]:
train_ds = CharCNNDataset(train_df, stoi=stoi, max_len=MAX_LEN, numeric_features=None)
dev_ds = CharCNNDataset(dev_df, stoi=stoi, max_len=MAX_LEN, numeric_features=None)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
dev_loader = DataLoader(dev_ds, batch_size=256, shuffle=False)

In [11]:
class CharCNN(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=64,
        num_filters=128,
        kernel_sizes=(3, 5, 7),
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k)
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), 1)

    def forward(self, x):
        # x: [batch, seq_len]
        emb = self.embedding(x)
        emb = emb.transpose(1, 2)

        conv_outs = []
        for conv in self.convs:
            h = torch.relu(conv(emb))
            h = torch.max(h, dim=2).values   #max pooling
            conv_outs.append(h)

        h = torch.cat(conv_outs, dim=1)
        h = self.dropout(h)
        logits = self.fc(h).squeeze(1)
        return logits

In [12]:
def evaluate_char_model(model, loader, device, threshold=0.5):
    model.eval()

    all_probs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            probs = torch.sigmoid(logits)
            preds = (probs >= threshold).long()

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    report = classification_report(all_labels, all_preds, digits=4)
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    f1_pos = f1_score(all_labels, all_preds, average="binary")

    return {
        "report": report,
        "f1_macro": f1_macro,
        "f1_pos": f1_pos,
        "probs": np.array(all_probs),
        "preds": np.array(all_preds),
        "labels": np.array(all_labels),
    }

In [13]:
def train_char_model(
    model,
    train_loader,
    dev_loader,
    device,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=10
):
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    best_state = None
    best_dev_f1 = -1

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        dev_metrics = evaluate_char_model(model, dev_loader, device)

        print(
            f"epoch {epoch:02d} | "
            f"train_loss={avg_train_loss:.4f} | "
            f"dev_macro_f1={dev_metrics['f1_macro']:.4f} | "
            f"dev_pos_f1={dev_metrics['f1_pos']:.4f}"
        )

        if dev_metrics["f1_macro"] > best_dev_f1:
            best_dev_f1 = dev_metrics["f1_macro"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model

In [14]:
char_cnn = CharCNN(
    vocab_size=len(stoi),
    embed_dim=64,
    num_filters=128,
    kernel_sizes=(3, 5, 7),
    dropout=0.3
)

char_cnn = train_char_model(
    model=char_cnn,
    train_loader=train_loader,
    dev_loader=dev_loader,
    device=device,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=20
)

epoch 01 | train_loss=0.6795 | dev_macro_f1=0.5854 | dev_pos_f1=0.5593
epoch 02 | train_loss=0.6518 | dev_macro_f1=0.6032 | dev_pos_f1=0.4852
epoch 03 | train_loss=0.6426 | dev_macro_f1=0.6133 | dev_pos_f1=0.5678
epoch 04 | train_loss=0.6309 | dev_macro_f1=0.5384 | dev_pos_f1=0.3208
epoch 05 | train_loss=0.6235 | dev_macro_f1=0.5155 | dev_pos_f1=0.2732
epoch 06 | train_loss=0.6244 | dev_macro_f1=0.6137 | dev_pos_f1=0.5051
epoch 07 | train_loss=0.6191 | dev_macro_f1=0.5830 | dev_pos_f1=0.4202
epoch 08 | train_loss=0.6184 | dev_macro_f1=0.4524 | dev_pos_f1=0.1386
epoch 09 | train_loss=0.6156 | dev_macro_f1=0.5492 | dev_pos_f1=0.3404
epoch 10 | train_loss=0.6147 | dev_macro_f1=0.5105 | dev_pos_f1=0.2543
epoch 11 | train_loss=0.6182 | dev_macro_f1=0.5489 | dev_pos_f1=0.3367
epoch 12 | train_loss=0.6145 | dev_macro_f1=0.3997 | dev_pos_f1=0.0327
epoch 13 | train_loss=0.6106 | dev_macro_f1=0.6064 | dev_pos_f1=0.5652
epoch 14 | train_loss=0.6137 | dev_macro_f1=0.6185 | dev_pos_f1=0.5510
epoch 

In [15]:
metrics = evaluate_char_model(char_cnn, dev_loader, device)

print("=" * 60)
print("PURE CHAR CNN PERFORMANCE")
print("=" * 60)
print(f"F1 Macro: {metrics['f1_macro']:.4f}")
print(f"F1 Positive: {metrics['f1_pos']:.4f}")
print(metrics["report"])

PURE CHAR CNN PERFORMANCE
F1 Macro: 0.6185
F1 Positive: 0.5510
              precision    recall  f1-score   support

         0.0     0.7244    0.6513    0.6859      3120
         1.0     0.5121    0.5963    0.5510      1915

    accuracy                         0.6304      5035
   macro avg     0.6183    0.6238    0.6185      5035
weighted avg     0.6437    0.6304    0.6346      5035



In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_feat_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_feat),
    columns=X_train_feat.columns,
    index=X_train_feat.index
)

X_dev_feat_scaled = pd.DataFrame(
    scaler.transform(X_dev_feat),
    columns=X_dev_feat.columns,
    index=X_dev_feat.index
)

In [17]:
train_ds_hybrid = CharCNNDataset(
    train_df,
    stoi=stoi,
    max_len=MAX_LEN,
    numeric_features=X_train_feat_scaled
)

dev_ds_hybrid = CharCNNDataset(
    dev_df,
    stoi=stoi,
    max_len=MAX_LEN,
    numeric_features=X_dev_feat_scaled
)

train_loader_hybrid = DataLoader(train_ds_hybrid, batch_size=128, shuffle=True)
dev_loader_hybrid = DataLoader(dev_ds_hybrid, batch_size=256, shuffle=False)

In [18]:
class CharCNNWithFeatures(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_numeric_features,
        embed_dim=64,
        num_filters=128,
        kernel_sizes=(3, 5, 7),
        hidden_dim=128,
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k)
            for k in kernel_sizes
        ])

        cnn_dim = num_filters * len(kernel_sizes)

        self.num_proj = nn.Sequential(
            nn.Linear(num_numeric_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.head = nn.Sequential(
            nn.Linear(cnn_dim + hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x_char, x_num):
        emb = self.embedding(x_char)
        emb = emb.transpose(1, 2)

        conv_outs = []
        for conv in self.convs:
            h = torch.relu(conv(emb))
            h = torch.max(h, dim=2).values
            conv_outs.append(h)

        h_char = torch.cat(conv_outs, dim=1)
        h_num = self.num_proj(x_num)

        h = torch.cat([h_char, h_num], dim=1)
        logits = self.head(h).squeeze(1)
        return logits

In [19]:
def evaluate_hybrid_model(model, loader, device, threshold=0.5):
    model.eval()

    all_probs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x_char, x_num, yb in loader:
            x_char = x_char.to(device)
            x_num = x_num.to(device)
            yb = yb.to(device)

            logits = model(x_char, x_num)
            probs = torch.sigmoid(logits)
            preds = (probs >= threshold).long()

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    report = classification_report(all_labels, all_preds, digits=4)
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    f1_pos = f1_score(all_labels, all_preds, average="binary")

    return {
        "report": report,
        "f1_macro": f1_macro,
        "f1_pos": f1_pos,
        "probs": np.array(all_probs),
        "preds": np.array(all_preds),
        "labels": np.array(all_labels),
    }

In [20]:
def train_hybrid_model(
    model,
    train_loader,
    dev_loader,
    device,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=10
):
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    best_state = None
    best_dev_f1 = -1

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for x_char, x_num, yb in train_loader:
            x_char = x_char.to(device)
            x_num = x_num.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            logits = model(x_char, x_num)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        dev_metrics = evaluate_hybrid_model(model, dev_loader, device)

        print(
            f"epoch {epoch:02d} | "
            f"train_loss={avg_train_loss:.4f} | "
            f"dev_macro_f1={dev_metrics['f1_macro']:.4f} | "
            f"dev_pos_f1={dev_metrics['f1_pos']:.4f}"
        )

        if dev_metrics["f1_macro"] > best_dev_f1:
            best_dev_f1 = dev_metrics["f1_macro"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model

In [21]:
hybrid_model = CharCNNWithFeatures(
    vocab_size=len(stoi),
    num_numeric_features=X_train_feat_scaled.shape[1],
    embed_dim=64,
    num_filters=128,
    kernel_sizes=(3, 5, 7),
    hidden_dim=128,
    dropout=0.3
)

hybrid_model = train_hybrid_model(
    model=hybrid_model,
    train_loader=train_loader_hybrid,
    dev_loader=dev_loader_hybrid,
    device=device,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=20
)

epoch 01 | train_loss=0.2725 | dev_macro_f1=0.9150 | dev_pos_f1=0.8954
epoch 02 | train_loss=0.1707 | dev_macro_f1=0.9146 | dev_pos_f1=0.8891
epoch 03 | train_loss=0.1506 | dev_macro_f1=0.9366 | dev_pos_f1=0.9200
epoch 04 | train_loss=0.1378 | dev_macro_f1=0.9409 | dev_pos_f1=0.9260
epoch 05 | train_loss=0.1196 | dev_macro_f1=0.9312 | dev_pos_f1=0.9163
epoch 06 | train_loss=0.1056 | dev_macro_f1=0.9351 | dev_pos_f1=0.9203
epoch 07 | train_loss=0.0876 | dev_macro_f1=0.9348 | dev_pos_f1=0.9163
epoch 08 | train_loss=0.0801 | dev_macro_f1=0.9457 | dev_pos_f1=0.9323
epoch 09 | train_loss=0.0677 | dev_macro_f1=0.9138 | dev_pos_f1=0.8976
epoch 10 | train_loss=0.0633 | dev_macro_f1=0.9421 | dev_pos_f1=0.9274
epoch 11 | train_loss=0.0521 | dev_macro_f1=0.9444 | dev_pos_f1=0.9302
epoch 12 | train_loss=0.0489 | dev_macro_f1=0.9406 | dev_pos_f1=0.9245
epoch 13 | train_loss=0.0467 | dev_macro_f1=0.9222 | dev_pos_f1=0.9068
epoch 14 | train_loss=0.0640 | dev_macro_f1=0.9425 | dev_pos_f1=0.9291
epoch 

In [22]:
hybrid_metrics = evaluate_hybrid_model(hybrid_model, dev_loader_hybrid, device)

print("=" * 60)
print("CHAR CNN + FEATURES PERFORMANCE")
print("=" * 60)
print(f"F1 Macro: {hybrid_metrics['f1_macro']:.4f}")
print(f"F1 Positive: {hybrid_metrics['f1_pos']:.4f}")
print(hybrid_metrics["report"])

CHAR CNN + FEATURES PERFORMANCE
F1 Macro: 0.9457
F1 Positive: 0.9323
              precision    recall  f1-score   support

         0.0     0.9540    0.9641    0.9590      3120
         1.0     0.9405    0.9243    0.9323      1915

    accuracy                         0.9490      5035
   macro avg     0.9473    0.9442    0.9457      5035
weighted avg     0.9489    0.9490    0.9489      5035



In [23]:
test_df = pd.read_csv("/content/nlp-sentenceSplitter/data/processed/UD_English-EWT/en_ewt-ud-test.csv")
print(test_df.shape)
print(test_df["label"].value_counts())

(4994, 9)
label
0    3031
1    1963
Name: count, dtype: int64


In [24]:
X_test_feat = feat_extractor.transform(test_df)
X_test_feat_scaled = pd.DataFrame(
    scaler.transform(X_test_feat),
    columns=X_test_feat.columns,
    index=X_test_feat.index
)

test_ds = CharCNNDataset(
    test_df,
    stoi=stoi,
    max_len=MAX_LEN,
    numeric_features=None
)

test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

test_ds_hybrid = CharCNNDataset(
    test_df,
    stoi=stoi,
    max_len=MAX_LEN,
    numeric_features=X_test_feat_scaled
)

test_loader_hybrid = DataLoader(test_ds_hybrid, batch_size=256, shuffle=False)

In [25]:
test_metrics_pure = evaluate_char_model(char_cnn, test_loader, device)

print("=" * 60)
print("PURE CHAR CNN ON TEST")
print("=" * 60)
print(f"F1 Macro: {test_metrics_pure['f1_macro']:.4f}")
print(f"F1 Positive: {test_metrics_pure['f1_pos']:.4f}")
print(test_metrics_pure["report"])

PURE CHAR CNN ON TEST
F1 Macro: 0.6111
F1 Positive: 0.5526
              precision    recall  f1-score   support

         0.0     0.7087    0.6348    0.6697      3031
         1.0     0.5143    0.5970    0.5526      1963

    accuracy                         0.6199      4994
   macro avg     0.6115    0.6159    0.6111      4994
weighted avg     0.6322    0.6199    0.6236      4994



In [26]:
test_metrics_hybrid = evaluate_hybrid_model(hybrid_model, test_loader_hybrid, device)

print("=" * 60)
print("CHAR CNN + FEATURES ON TEST")
print("=" * 60)
print(f"F1 Macro: {test_metrics_hybrid['f1_macro']:.4f}")
print(f"F1 Positive: {test_metrics_hybrid['f1_pos']:.4f}")
print(test_metrics_hybrid["report"])

CHAR CNN + FEATURES ON TEST
F1 Macro: 0.9342
F1 Positive: 0.9191
              precision    recall  f1-score   support

         0.0     0.9367    0.9624    0.9494      3031
         1.0     0.9394    0.8996    0.9191      1963

    accuracy                         0.9377      4994
   macro avg     0.9380    0.9310    0.9342      4994
weighted avg     0.9378    0.9377    0.9375      4994



In [27]:
test_paths = {
    "GUM": "/content/nlp-sentenceSplitter/data/processed/UD_English-GUM/en_gum-ud-test.csv",
    "ParTUT": "/content/nlp-sentenceSplitter/data/processed/UD_English-ParTUT/en_partut-ud-test.csv",
    "PUD": "/content/nlp-sentenceSplitter/data/processed/UD_English-PUD/en_pud-ud-test.csv",
}

In [28]:
def evaluate_on_dataset(
    df,
    model,
    feat_extractor,
    scaler,
    stoi,
    max_len,
    device,
):
    y = df["label"].astype(int)
    X_feat = feat_extractor.transform(df)

    X_feat_scaled = pd.DataFrame(
        scaler.transform(X_feat),
        columns=X_feat.columns,
        index=X_feat.index
    )

    ds = CharCNNDataset(
        df,
        stoi=stoi,
        max_len=max_len,
        numeric_features=X_feat_scaled
    )

    loader = DataLoader(ds, batch_size=256, shuffle=False)
    metrics = evaluate_hybrid_model(model, loader, device)

    return metrics

In [29]:
results = {}

for name, path in test_paths.items():
    df = pd.read_csv(path)

    metrics = evaluate_on_dataset(
        df=df,
        model=hybrid_model,
        feat_extractor=feat_extractor,
        scaler=scaler,
        stoi=stoi,
        max_len=MAX_LEN,
        device=device,
    )

    results[name] = metrics

    print("\n" + "=" * 60)
    print(f"{name} DATASET")
    print("=" * 60)
    print(f"F1 Macro: {metrics['f1_macro']:.4f}")
    print(f"F1 Positive: {metrics['f1_pos']:.4f}")
    print(metrics["report"])


GUM DATASET
F1 Macro: 0.9608
F1 Positive: 0.9493
              precision    recall  f1-score   support

         0.0     0.9744    0.9701    0.9722      2672
         1.0     0.9454    0.9532    0.9493      1454

    accuracy                         0.9641      4126
   macro avg     0.9599    0.9616    0.9608      4126
weighted avg     0.9642    0.9641    0.9642      4126


ParTUT DATASET
F1 Macro: 0.9946
F1 Positive: 0.9934
              precision    recall  f1-score   support

         0.0     0.9916    1.0000    0.9958       237
         1.0     1.0000    0.9869    0.9934       153

    accuracy                         0.9949       390
   macro avg     0.9958    0.9935    0.9946       390
weighted avg     0.9949    0.9949    0.9949       390


PUD DATASET
F1 Macro: 0.9922
F1 Positive: 0.9896
              precision    recall  f1-score   support

         0.0     0.9990    0.9908    0.9949      2064
         1.0     0.9813    0.9980    0.9896      1000

    accuracy                 